In [5]:
import sys
sys.path.append("/Users/emilieyu/endotehelial-masboss")

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import copy

from abm_v2.membrane_node import MembraneNode
from abm_v2.flow_field import FlowField
from abm_v2.spring import Spring
from abm_v2.stress_fibre import StressFibre
from abm_v2.endothelial_cell import EndothelialCell
from abm.rho_lookup_table import RhoLookupTable
from src.config import load_abm_sim_cfg
from src.paths import BM_RESULTS_DIR

CFG = load_abm_sim_cfg()
lut = RhoLookupTable(CFG, BM_RESULTS_DIR)
flow = FlowField()


>>> DEBUG: Successfully loaded recruitment parameter sweep data.
>>> DEBUG: Successfully built interpolators
LUT ready | rest: RhoA=0.463 RhoC=0.437


### Node Test

In [1]:


# Pole node — high f_normal
n_pole = MembraneNode(0, np.array([12.0, 0.0]), lut, CFG)
n_pole.f_normal = CFG['mechanics']['f_magnitude']
n_pole.update_signalling()
print(f"Pole node:  f_normal={n_pole.f_normal:.3f} "
      f"P_RhoA={n_pole.P_RhoA:.3f} P_RhoC={n_pole.P_RhoC:.3f}")

# Flank node — zero f_normal
n_flank = MembraneNode(1, np.array([0.0, 12.0]), lut, CFG)
n_flank.f_normal = 0.0
n_flank.update_signalling()
print(f"Flank node: f_normal={n_flank.f_normal:.3f} "
      f"P_RhoA={n_flank.P_RhoA:.3f} P_RhoC={n_flank.P_RhoC:.3f}")

# Pole should have higher RhoA than flank
assert n_pole.P_RhoA > n_flank.P_RhoA, "Pole should have higher RhoA"
assert n_pole.P_RhoC > n_flank.P_RhoC, "Pole should have higher RhoC"
print("MembraneNode signalling: PASS")

NameError: name 'MembraneNode' is not defined

### Spring Test

In [ ]:
flow = np.array([1.0, 0.0])

n1 = MembraneNode(0, np.array([0.0, 0.0]), lut, CFG)
n2 = MembraneNode(1, np.array([5.0, 0.0]), lut, CFG)
s  = Spring(0, n1, n2, rest_length=5.0,
            k_cortex=CFG['mechanics']['k_cortex'],
            lut=lut, cfg=CFG)
s._init_alignment = 1.0

# Test 1 — at rest, no RhoA
s.update_geometry(flow)
s.update_cortex()
print(f"At rest:    T={s.tension:.4f}  k={s.k_active:.4f}")
assert abs(s.tension) < 1e-6

# Test 2 — stretched, no RhoA
n2.pos = np.array([7.0, 0.0])
s.update_geometry(flow)
s.update_tension()
print(f"Stretched:  T={s.tension:.4f}  k={s.k_active:.4f}")
assert s.tension > 0

# Test 3 — stretched, high RhoA at both nodes
n1.P_RhoA = 0.7
n2.P_RhoA = 0.7
s.update_geometry(flow)
s.update_tension()
print(f"High RhoA:  T={s.tension:.4f}  k={s.k_active:.4f}")
assert s.k_active > CFG['mechanics']['k_cortex']

# Test 4 — asymmetric RhoA
n1.P_RhoA = 0.7
n2.P_RhoA = lut.rhoa_rest
s.update_geometry(flow)
s.update_tension()
k_asym = s.k_active
n1.P_RhoA = 0.7
n2.P_RhoA = 0.7
s.update_geometry(flow)
s.update_tension()
k_high = s.k_active
print(f"Asymmetric k={k_asym:.4f}  both-high k={k_high:.4f}  "
      f"(asymmetric should be between baseline and both-high)")
assert CFG['mechanics']['k_cortex'] < k_asym < k_high

# Test 5 — force application
n1.force = np.zeros(2)
n2.force = np.zeros(2)
n2.pos = np.array([7.0, 0.0])
s.update_geometry(flow)
s.update_cortex()
s.apply_forces()
print(f"Node1 force: {n1.force.round(4)}")
print(f"Node2 force: {n2.force.round(4)}")
assert np.allclose(n1.force, -n2.force)

print("Spring: PASS")

AttributeError: 'Spring' object has no attribute 'update_cortex'

### Stress Fibre Tests

In [3]:

up = MembraneNode(0, np.array([-12.0, 0.0]), lut, CFG)
dn = MembraneNode(1, np.array([ 12.0, 0.0]), lut, CFG)
sf = StressFibre(up, dn, CFG)

nodes = [
    MembraneNode(2, np.array([ 0.0,  12.0]), lut, CFG),  # top flank
    MembraneNode(3, np.array([ 0.0, -12.0]), lut, CFG),  # bottom flank
    MembraneNode(4, np.array([ 0.0,   6.0]), lut, CFG),  # mid flank
    MembraneNode(5, np.array([12.0,   0.0]), lut, CFG),  # pole
]

# Test 1 — no a_sf
sf.a_sf = 0.0
sf.update_geometry()
sf.update_tension()
assert sf.t_sf == 0.0
print(f"No a_sf: t_sf={sf.t_sf:.4f} ✓")

# Test 2 — full a_sf, cable tension
sf.a_sf = 1.0
sf.update_geometry()
sf.update_tension()
print(f"Full a_sf: L={sf.L_current:.3f}  t_sf={sf.t_sf:.4f}")
assert sf.t_sf == sf.k_sf * 1.0 * sf.L_current

# Test 3 — cable forces
up.force = np.zeros(2)
dn.force = np.zeros(2)
sf.apply_forces()
assert np.allclose(up.force, -dn.force)
assert up.force[0] > 0
assert dn.force[0] < 0
print(f"Cable forces: up={up.force.round(3)}  dn={dn.force.round(3)} ✓")

# Test 4 — squeeze
max_y = max(abs(n.pos[1] - sf.cable_y) for n in nodes)
for node in nodes:
    f = sf.get_squeeze_force(node.pos[1], max_y)
    node.apply_force(np.array([0.0, f]))
    print(f"  y={node.pos[1]:+6.1f}  squeeze={f:+.4f}")

assert nodes[0].force[1] < 0      # top flank pushed down
assert nodes[1].force[1] > 0      # bottom flank pushed up
assert abs(nodes[3].force[1]) < 1e-6  # pole: no squeeze
assert abs(nodes[0].force[1]) > abs(nodes[2].force[1])  # flank > mid

print("StressFibre: PASS")

No a_sf: t_sf=0.0000 ✓
Full a_sf: L=24.000  t_sf=24.0000
Cable forces: up=[24.  0.]  dn=[-24.   0.] ✓
  y= +12.0  squeeze=-0.3000
  y= -12.0  squeeze=+0.3000
  y=  +6.0  squeeze=-0.1500
  y=  +0.0  squeeze=-0.0000
StressFibre: PASS


In [5]:
np.mean([n_pole.P_RhoA, n_flank.P_RhoA])

np.float64(0.5535)

### Node Initialisation Test

In [2]:
cell = EndothelialCell(0, np.array([0.0, 0.0]),
                       lut=lut, cfg=CFG,
                       n_nodes=CFG['sim']['n_nodes'],
                       radius=CFG['sim']['radius'])

In [4]:

# Node classification
roles = [n.role for n in cell.nodes]
print(f"upstream={roles.count('upstream')}  "
      f"downstream={roles.count('downstream')}  "
      f"lateral={roles.count('lateral')}")
assert roles.count('upstream')   == 3
assert roles.count('downstream') == 3
assert roles.count('lateral')    == 10

# FA nodes
print(f"FA nodes: {len(cell.fa_nodes)} "
      f"(ids: {list(cell.fa_nodes.keys())})")
assert len(cell.fa_nodes) == 2

# SF cables
print(f"SF cables: {len(cell.stress_fibres)}")
for sf in cell.stress_fibres:
    print(f"  {sf}")
assert len(cell.stress_fibres) == 1

# Area
print(f"Area: {cell.target_area:.1f} units² = "
      f"{cell.target_area*9:.0f} µm²")

print("EndothelialCell init: PASS")

upstream=3  downstream=3  lateral=10
FA nodes: 2 (ids: [4, 12])
SF cables: 1
  StressFibre(L=0.000 | contractility=0.000 | T_sf=0.0000 | cable_y=0.000)
Area: 440.9 units² = 3968 µm²
EndothelialCell init: PASS


### Node Force Methods

In [6]:

flow = FlowField(magnitude=CFG['mechanics']['f_magnitude'])
cell = EndothelialCell(0, np.array([0.0, 0.0]),
                       lut=lut, cfg=CFG,
                       n_nodes=CFG['sim']['n_nodes'],
                       radius=CFG['sim']['radius'])

# Test 1 — shear: no translation, f_normal highest at poles
cell._apply_shear(flow)
net = sum(n.force for n in cell.nodes)
print(f"Net force after shear: {net.round(6)}")
assert np.allclose(net, 0, atol=1e-6), "Net force must be zero"

pole_fn   = np.mean([n.f_normal for n in cell.nodes
                     if n.role in ('upstream','downstream')])
flank_fn  = np.mean([n.f_normal for n in cell.nodes
                     if n.role == 'lateral'])
print(f"Pole f_normal={pole_fn:.3f}  Flank f_normal={flank_fn:.3f}")
assert pole_fn > flank_fn, "Poles should have higher f_normal"

# Reset forces
for n in cell.nodes: n.force = np.zeros(2)

# Test 2 — FA anchoring: pole nodes at rest position → zero force
cell._apply_fa_anchoring()
pole_forces = [np.linalg.norm(cell.nodes[i].force)
               for i in range(cell.n_nodes)
               if cell.nodes[i].id in cell.fa_nodes]
print(f"FA forces at rest: {[round(f,6) for f in pole_forces]}")
assert all(f < 1e-6 for f in pole_forces), "No FA force when at rest pos"

# Reset
for n in cell.nodes: n.force = np.zeros(2)

# Test 3 — SF squeeze: flanks squeezed, poles unaffected
for sf in cell.stress_fibres:
    sf.a_sf = 1.0
    sf.update_geometry_and_tension()

cell._apply_sf_squeeze()
lateral_y = [abs(cell.nodes[i].force[1])
             for i in range(cell.n_nodes)
             if cell.nodes[i].role == 'lateral']
pole_y    = [abs(cell.nodes[i].force[1])
             for i in range(cell.n_nodes)
             if cell.nodes[i].role in ('upstream','downstream')]
print(f"Mean lateral squeeze: {np.mean(lateral_y):.4f}")
print(f"Mean pole squeeze:    {np.mean(pole_y):.4f}")
assert np.mean(lateral_y) > np.mean(pole_y), \
    "Lateral nodes should feel more squeeze than poles"

print("Force methods: PASS")

Net force after shear: [0. 0.]
Pole f_normal=2.848  Flank f_normal=1.308
FA forces at rest: [np.float64(0.0), np.float64(0.0)]
Mean lateral squeeze: 0.2557
Mean pole squeeze:    0.0765
Force methods: PASS


0.65

### Simulation Test

In [7]:
flow = FlowField(magnitude=CFG['mechanics']['f_magnitude'])
cell = EndothelialCell(0, np.array([0.0, 0.0]),
                       lut=lut, cfg=CFG,
                       n_nodes=CFG['sim']['n_nodes'],
                       radius=CFG['sim']['radius'])
print("\nRunning 100 steps...")
for step in range(100):
    cell.step(flow, dt=CFG['sim']['dt'])
    if step % 20 == 0:
        s = cell.get_state()
        print(f"  step {step:>4}: ar={s['ar']:.3f}  "
              f"orient={s['orientation']:.1f}°  "
              f"a_sf={s['a_sf']:.4f}  "
              f"RhoA={s['mean_rhoa']:.3f}  "
              f"RhoC={s['mean_rhoc']:.3f}  "
              f"area={s['area_err']:.3f}")

s = cell.get_state()
print(f"\nFinal: {cell}")
assert s['ar'] > 1.0,           "Cell should elongate"
assert abs(s['orientation']) < 45 or abs(s['orientation']) > 135, \
    "Elongation should be along flow axis"
assert 0.9 < s['area_err'] < 1.1, "Area should be conserved"
print("Full simulation: PASS")


Running 100 steps...
  step    0: ar=1.000  orient=0.0°  a_sf=0.0016  RhoA=0.590  RhoC=0.610  area=1.000
  step   20: ar=1.002  orient=90.0°  a_sf=0.0320  RhoA=0.590  RhoC=0.610  area=0.173
  step   40: ar=1.022  orient=-180.0°  a_sf=0.0597  RhoA=0.605  RhoC=0.602  area=0.259
  step   60: ar=1.029  orient=90.0°  a_sf=0.0853  RhoA=0.603  RhoC=0.598  area=0.879
  step   80: ar=1.027  orient=90.0°  a_sf=0.1089  RhoA=0.605  RhoC=0.602  area=0.879

Final: EndothelialCell(id=0 | ar=1.02 | area_err=0.996 | a_sf=0.130 | RhoA=0.605 RhoC=0.602)


AssertionError: Elongation should be along flow axis

### Sim Perturbation Tests

In [ ]:
import copy

conditions = {
    'WT':            {},
    'DSP_KO':        {'DSP':  {'knocked_out': True}},
    'TJP1_KO':       {'TJP1': {'knocked_out': True}},
    'JCAD_KO': {'JCAD': {'knocked_out': True}}
    'DSP_JCAD_DKO':  {'DSP':  {'knocked_out': True},
                      'JCAD': {'knocked_out': True}},
    'TJP1_JCAD_DKO': {'TJP1': {'knocked_out': True},
                      'JCAD': {'knocked_out': True}},
}

def run_condition(cfg, lut, knockouts, n_steps=500):
    c = copy.deepcopy(cfg)
    for protein, overrides in knockouts.items():
        c['hill_params'][protein].update(overrides)
    cell = EndothelialCell(0, np.array([0.0, 0.0]),
                           lut=lut, cfg=c,
                           n_nodes=c['sim']['n_nodes'],
                           radius=c['sim']['radius'])
    flow = FlowField(magnitude=c['mechanics']['f_magnitude'])
    for _ in range(n_steps):
        cell.step(flow, dt=c['sim']['dt'])
    return cell.get_state()

print(f"{'Condition':<18} {'AR':>6} {'orient':>8} "
      f"{'a_sf':>6} {'RhoA':>6} {'RhoC':>6} {'area':>6}")
print("-" * 60)

results = {}
for name, knockouts in conditions.items():
    s = run_condition(CFG, lut, knockouts, n_steps=500)
    results[name] = s
    print(f"{name:<18} {s['ar']:>6.3f} {s['orientation']:>8.1f}° "
          f"{s['a_sf']:>6.3f} {s['mean_rhoa']:>6.3f} "
          f"{s['mean_rhoc']:>6.3f} {s['area_err']:>6.3f}")

# Check primary ordering
print("\nOrdering checks:")
print(f"TJP1_KO < WT:    {results['TJP1_KO']['ar']:.3f} < {results['WT']['ar']:.3f} "
      f"→ {'✓' if results['TJP1_KO']['ar'] < results['WT']['ar'] else '✗'}")
print(f"WT < DSP_KO:     {results['WT']['ar']:.3f} < {results['DSP_KO']['ar']:.3f} "
      f"→ {'✓' if results['WT']['ar'] < results['DSP_KO']['ar'] else '✗'}")
print(f"TJP1_KO < TJP1_JCAD: {results['TJP1_KO']['ar']:.3f} < "
      f"{results['TJP1_JCAD_DKO']['ar']:.3f} "
      f"→ {'✓' if results['TJP1_KO']['ar'] < results['TJP1_JCAD_DKO']['ar'] else '✗'}")
print(f"TJP1_JCAD < WT:  {results['TJP1_JCAD_DKO']['ar']:.3f} < {results['WT']['ar']:.3f} "
      f"→ {'✓' if results['TJP1_JCAD_DKO']['ar'] < results['WT']['ar'] else '✗'}")

Condition              AR   orient   a_sf   RhoA   RhoC   area
------------------------------------------------------------
WT                  2.508   -180.0°  0.414  0.500  0.623  1.002
DSP_KO              2.611      0.0°  0.616  0.314  0.700  1.001
TJP1_KO             2.441   -180.0°  0.000  0.617  0.361  0.998
DSP_JCAD_DKO        2.674   -180.0°  0.376  0.367  0.606  1.005
TJP1_JCAD_DKO       2.537   -180.0°  0.000  0.532  0.410  0.999

Ordering checks:
TJP1_KO < WT:    2.441 < 2.508 → ✓
WT < DSP_KO:     2.508 < 2.611 → ✓
TJP1_KO < TJP1_JCAD: 2.441 < 2.537 → ✓
TJP1_JCAD < WT:  2.537 < 2.508 → ✗


## DEBUG

In [7]:
cfg_drag = copy.deepcopy(CFG)
cfg_drag['mechanics']['rhoa_k_gain'] = 0.0   # no RhoA stiffening
cfg_drag['mechanics']['k_sf']        = 0.0   # no SF
cfg_drag['mechanics']['nu_sf']       = 0.0   # no squeeze
cfg_drag['mechanics']['k_fa']        = 0.0   # no FA
cfg_drag['mechanics']['drag_fraction'] = 0.1

cell_drag = EndothelialCell(0, np.array([0.0, 0.0]),
                             lut=lut, cfg=cfg_drag,
                             n_nodes=cfg_drag['sim']['n_nodes'],
                             radius=cfg_drag['sim']['radius'])
flow = FlowField(magnitude=cfg_drag['mechanics']['f_magnitude'])

print("Drag only — no signalling, no SF")
print(f"{'step':>6} {'AR':>6} {'orient':>8} {'area':>6}")
for step in range(500):
    cell_drag.step(flow, dt=cfg_drag['sim']['dt'])
    if step % 100 == 0 or step == 499:
        s = cell_drag.get_state()
        print(f"{step:>6} {s['ar']:>6.3f} "
              f"{s['orientation']:>8.1f}° "
              f"{s['area_err']:>6.3f}")

Drag only — no signalling, no SF
  step     AR   orient   area
     0  1.501      0.0°  1.000
   100  1.209    180.0°  1.007
   200  1.297    180.0°  1.000
   300  1.384    180.0°  1.000
   400  1.463    180.0°  1.000
   499  1.533    180.0°  1.000


In [5]:

flow = FlowField(magnitude=CFG['mechanics']['f_magnitude'])

# Config with SF disabled and uniform cortex (no RhoA effect)
cfg_test = copy.deepcopy(CFG)
cfg_test['mechanics']['k_sf']      = 0.0   # no SF cable tension
cfg_test['mechanics']['nu_sf']     = 0.0   # no squeeze
cfg_test['mechanics']['k_fa']      = 0.0   # no FA anchoring
cfg_test['mechanics']['rhoa_k_gain'] = 0.0 # uniform cortex — no RhoA stiffening

cell1 = EndothelialCell(0, np.array([0.0, 0.0]),
                        lut=lut, cfg=cfg_test,
                        n_nodes=CFG['sim']['n_nodes'],
                        radius=CFG['sim']['radius'])

print("Test 1 — shear only, uniform cortex")
print(f"{'step':>6} {'AR':>6} {'orient':>8} {'area':>6}")
for step in range(500):
    cell1.step(flow, dt=CFG['sim']['dt'])
    if step % 100 == 0 or step == 499:
        s = cell1.get_state()
        print(f"{step:>6} {s['ar']:>6.3f} {s['orientation']:>8.1f}° "
              f"{s['area_err']:>6.3f}")

Test 1 — shear only, uniform cortex
  step     AR   orient   area
     0  1.500      0.0°  1.000
   100  1.106    180.0°  1.001
   200  1.023    180.0°  1.006
   300  1.005     90.0°  1.013
   400  1.008    180.0°  1.003
   499  1.056     90.0°  1.018


In [6]:
cfg_test1 = copy.deepcopy(CFG)
cfg_test1['mechanics']['rhoa_k_gain'] = 0.0  # uniform cortex

cell1 = EndothelialCell(0, np.array([0.0, 0.0]),
                        lut=lut, cfg=cfg_test1,
                        n_nodes=CFG['sim']['n_nodes'],
                        radius=CFG['sim']['radius'])

print("Test 1 — SF only, forced a_sf=0.8, uniform cortex")
print(f"{'step':>6} {'AR':>6} {'orient':>8} {'area':>6} "
      f"{'SF_t':>8} {'FA_f':>8} {'squeeze':>8}")

for step in range(500):
    # Force a_sf each step
    for sf in cell1.stress_fibres:
        sf.a_sf = 0.8

    cell1.step(flow, dt=CFG['sim']['dt'])

    if step % 100 == 0 or step == 499:
        s = cell1.get_state()

        # SF tension
        sf_t = cell1.stress_fibres[0].t_sf

        # FA force on centre pole node
        fa_node = cell1.nodes[4]
        fa_pos  = cell1.fa_nodes[fa_node.id]
        fa_f    = CFG['mechanics']['k_fa'] * np.linalg.norm(
            fa_node.pos - fa_pos
        )

        # Squeeze on most lateral node
        sf0    = cell1.stress_fibres[0]
        max_y  = max(abs(n.pos[1] - sf0.cable_y) for n in cell1.nodes)
        sq_f   = abs(sf0.get_squeeze_force(
            cell1.nodes[0].pos[1], max_y
        ))

        print(f"{step:>6} {s['ar']:>6.3f} {s['orientation']:>8.1f}° "
              f"{s['area_err']:>6.3f} {sf_t:>8.3f} {fa_f:>8.3f} {sq_f:>8.4f}")

Test 1 — SF only, forced a_sf=0.8, uniform cortex
  step     AR   orient   area     SF_t     FA_f  squeeze
     0  1.493      0.0°  0.994    9.406    1.568   0.0004
   100  1.525   -180.0°  1.035    8.273    8.741   0.0376
   200  1.433   -180.0°  0.997    8.354    8.243   0.0667
   300  1.365   -180.0°  0.999    8.360    8.207   0.0855
   400  1.350   -180.0°  1.004    8.378    8.090   0.0987
   499  1.359   -180.0°  1.000    8.365    8.126   0.1054


In [8]:
cfg_test2 = copy.deepcopy(CFG)
cfg_test2['mechanics']['rhoa_k_gain'] = 0.0

cell2 = EndothelialCell(0, np.array([0.0, 0.0]),
                        lut=lut, cfg=cfg_test2,
                        n_nodes=CFG['sim']['n_nodes'],
                        radius=CFG['sim']['radius'])

# One step only — check force balance at FA nodes
for sf in cell2.stress_fibres:
    sf.a_sf = 0.8
    sf.update_geometry_and_tension()
    sf.apply_forces()   # inward pull on FA nodes

# FA force before anchoring
sf_force_up = cell2.nodes[4].force.copy()
print(f"SF pull on upstream pole:   {sf_force_up.round(4)}")

cell2._apply_fa_anchoring()

# FA anchoring adds to same node
total_force_up = cell2.nodes[4].force.copy()
fa_force_up    = total_force_up - sf_force_up
print(f"FA anchor on upstream pole: {fa_force_up.round(4)}")
print(f"Net force on upstream pole: {total_force_up.round(4)}")
print(f"Cancellation ratio: {np.linalg.norm(total_force_up) / np.linalg.norm(sf_force_up):.3f}")
print(f"(1.0 = perfect cancellation, 0.0 = full cancellation)")

SF pull on upstream pole:   [ 9.406 -0.   ]
FA anchor on upstream pole: [0. 0.]
Net force on upstream pole: [ 9.406 -0.   ]
Cancellation ratio: 1.000
(1.0 = perfect cancellation, 0.0 = full cancellation)


In [10]:
cell3 = EndothelialCell(0, np.array([0.0, 0.0]),
                        lut=lut, cfg=CFG,
                        n_nodes=CFG['sim']['n_nodes'],
                        radius=CFG['sim']['radius'])

for sf in cell3.stress_fibres:
    sf.a_sf = 0.8
    sf.update_geometry_and_tension()

# Reset forces and apply squeeze only
for n in cell3.nodes:
    n.force = np.zeros(2)

cell3._apply_sf_squeeze()

print("\nTest 3 — squeeze forces (should be inward, zero at poles):")
print(f"{'id':>3} {'role':>12} {'pos_y':>8} {'fy':>8} {'direction':>12}")
for node in cell3.nodes:
    fy        = node.force[1]
    direction = 'inward ✓' if (
        (node.pos[1] > 0 and fy < 0) or
        (node.pos[1] < 0 and fy > 0) or
        abs(node.pos[1]) < 0.1
    ) else 'OUTWARD ✗'
    print(f"{node.id:>3} {node.role:>12} {node.pos[1]:>8.3f} "
          f"{fy:>8.4f} {direction:>12}")


Test 3 — squeeze forces (should be inward, zero at poles):
 id         role    pos_y       fy    direction
  0      lateral    9.798  -0.2400     inward ✓
  1      lateral    9.052  -0.2217     inward ✓
  2      lateral    6.928  -0.1697     inward ✓
  3     upstream    3.750  -0.0918     inward ✓
  4     upstream    0.000   0.0000     inward ✓
  5     upstream   -3.750   0.0918     inward ✓
  6      lateral   -6.928   0.1697     inward ✓
  7      lateral   -9.052   0.2217     inward ✓
  8      lateral   -9.798   0.2400     inward ✓
  9      lateral   -9.052   0.2217     inward ✓
 10      lateral   -6.928   0.1697     inward ✓
 11   downstream   -3.750   0.0918     inward ✓
 12   downstream   -0.000   0.0000     inward ✓
 13   downstream    3.750  -0.0918     inward ✓
 14      lateral    6.928  -0.1697     inward ✓
 15      lateral    9.052  -0.2217     inward ✓


In [11]:
print(f"FA pos: {cell1.fa_nodes}")

FA pos: {4: array([-1.46969385e+01,  1.19990391e-15]), 12: array([ 1.46969385e+01, -2.39980782e-15])}


In [13]:
flow = FlowField(magnitude=CFG['mechanics']['f_magnitude'])
cell = EndothelialCell(0, np.array([0.0, 0.0]),
                       lut=lut, cfg=CFG,
                       n_nodes=CFG['sim']['n_nodes'],
                       radius=CFG['sim']['radius'])
print("\nRunning 100 steps...")
for step in range(100):
    cell.step(flow, dt=CFG['sim']['dt'])
    if step % 20 == 0:
        s = cell.get_state()
        print(f"  step {step:>4}: ar={s['ar']:.3f}  "
              f"orient={s['orientation']:.1f}°  "
              f"a_sf={s['a_sf']:.4f}  "
              f"RhoA={s['mean_rhoa']:.3f}  "
              f"RhoC={s['mean_rhoc']:.3f}  "
              f"area={s['area_err']:.3f}")

s = cell.get_state()
# After running WT condition
print(f"mean P_RhoC: {np.mean([n.P_RhoC for n in cell.nodes]):.3f}")
print(f"rhoc_rest:   {lut.rhoc_rest:.3f}")
print(f"delta_rhoC:  {np.mean([n.P_RhoC for n in cell.nodes]) - lut.rhoc_rest:.3f}")
print(f"rhoc_max:    0.365")
print(f"a_sf target: {(np.mean([n.P_RhoC for n in cell.nodes]) - lut.rhoc_rest) / 0.365:.3f}")


Running 100 steps...
  step    0: ar=1.500  orient=0.0°  a_sf=0.0015  RhoA=0.565  RhoC=0.598  area=1.000
  step   20: ar=5.860  orient=0.0°  a_sf=0.0294  RhoA=0.509  RhoC=0.568  area=0.027
  step   40: ar=1.167  orient=0.0°  a_sf=0.0518  RhoA=0.521  RhoC=0.583  area=1.002
  step   60: ar=1.113  orient=0.0°  a_sf=0.0711  RhoA=0.545  RhoC=0.576  area=1.030
  step   80: ar=1.149  orient=0.0°  a_sf=0.0921  RhoA=0.556  RhoC=0.581  area=1.015
mean P_RhoC: 0.575
rhoc_rest:   0.437
delta_rhoC:  0.138
rhoc_max:    0.365
a_sf target: 0.379
